# Typed extraction and exact value identity

The Rust pipeline converts the input DataFrame into a `TypedTable` once
per profile. Most detectors operate on this shared representation rather
than repeatedly calling Polars. Extraction handles the source buffers;
the pipeline also uses Polars to construct a verified pivot's wide form.

`TypedCol` contains signed and unsigned integers, floats, borrowed strings,
owned categorical labels, booleans, dates, datetimes, nulls, and opaque
unsupported values. The representation preserves the information needed
for exact counting and comparisons before methods choose their own
numerical projections.

## Borrowed, chunked, and owned numeric values

Int64, UInt64, and Float64 columns use `NumericValues`:

- `Borrowed` holds a dense, non-null Arrow slice.
- `Chunked` borrows one or more buffers and their validity bitmaps. It
  also supports a single nullable chunk or floats requiring normalization.
  A row lookup locates its chunk and applies validity/canonicalization.
- `Owned` holds converted values when a source dtype needs a normalized
  copy, such as a narrower integer type.

String columns retain borrowed string views. Categorical and enum columns
materialize owned labels. High-cardinality string counting can therefore
sort borrowed values instead of allocating a separate copy of every label.

## Equality and numerical projections

Unsigned integers keep all 64 bits. Datetimes keep their original tick
values and time unit, so adjacent nanoseconds remain distinct. Dates use
days since the epoch. Display formatting is separate from equality;
formatted strings are not the keys used to verify dependencies.

Float NaN and infinities are treated as missing, and −0.0 is normalized
to +0.0. Arrow validity identifies nulls. A suspected sentinel code such
as −999 remains a non-null observation; a sentinel finding does not
rewrite the input.

Calendar methods project timestamps to days using the source unit and
floor division. Numeric statistics and model features use floating-point
projections. Full-input counting and comparison do not depend on those
potentially approximate model representations.

## The Python boundary

The binding imports a Polars DataFrame through pyo3-polars interchange and
releases the GIL during Rust profiling. The pipeline borrows or converts
source buffers as described above and returns an owned native report.
Python dictionary properties parse its JSON on first access; Markdown and
JSON renderings can also come directly from the native report. Accessing
the report does not trigger a later scan of the source DataFrame.

The [counting chapter](counting.ipynb) explains the adaptive exact-count
algorithms, while the [architecture chapter](architecture.ipynb) shows
where extraction, discovery, and report selection fit together.
